# Assignment: ANN Salary Regression

Use the same `Churn_Modelling.csv` dataset, but change the target to `EstimatedSalary`. Your goal is to practice how an ANN changes when the task changes from classification to regression.

## Required Changes
- Target: `EstimatedSalary`
- Remove `EstimatedSalary` from input features
- Output layer: 1 neuron with **linear** activation
- Loss: choose `mse` or `mae` and explain your choice
- Evaluation: MAE, RMSE and R2
- Save the model in `.keras` format
- Build and deploy a Streamlit salary prediction app

## Step 1 - Imports

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import  StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
import pickle

import tensorflow as tf
import tensorflow.keras as keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import (EarlyStopping, TensorBoard)
import datetime

## Step 2 - Load and inspect the dataset

In [ ]:
data = pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


## Step 3 - EDA and data quality checks

In [ ]:
# shape, info, describe, missing values, duplicates, salary distribution
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  str    
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  str    
 5   Gender           10000 non-null  str    
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), str(3)
memory usage: 1.2 MB


In [ ]:
data.isnull().sum()

RowNumber          0
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64

In [ ]:
data.describe()

,RowNumber,CustomerId,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
count,10000.00000,1.000000e+04,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000
mean,5000.50000,1.569094e+07,650.528800,38.921800,5.012800,76485.889288,1.530200,0.70550,0.515100,100090.239881,0.203700
std,2886.89568,7.193619e+04,96.653299,10.487806,2.892174,62397.405202,0.581654,0.45584,0.499797,57510.492818,0.402769
min,1.00000,1.556570e+07,350.000000,18.000000,0.000000,0.000000,1.000000,0.00000,0.000000,11.580000,0.000000
25%,2500.75000,1.562853e+07,584.000000,32.000000,3.000000,0.000000,1.000000,0.00000,0.000000,51002.110000,0.000000
50%,5000.50000,1.569074e+07,652.000000,37.000000,5.000000,97198.540000,1.000000,1.00000,1.000000,100193.915000,0.000000
75%,7500.25000,1.575323e+07,718.000000,44.000000,7.000000,127644.240000,2.000000,1.00000,1.000000,149388.247500,0.000000
max,10000.00000,1.581569e+07,850.000000,92.000000,10.000000,250898.090000,4.000000,1.00000,1.000000,199992.480000,1.000000


In [ ]:
data.duplicated().sum()

np.int64(0)

In [ ]:
data['EstimatedSalary'].describe()

count     10000.000000
mean     100090.239881
std       57510.492818
min          11.580000
25%       51002.110000
50%      100193.915000
75%      149388.247500
max      199992.480000
Name: EstimatedSalary, dtype: float64

In [ ]:
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)   
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [ ]:
encod_gender= LabelEncoder()
data['Gender']= encod_gender.fit_transform(data['Gender'])
data.head(15
          )

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0
5,645,Spain,1,44,8,113755.78,2,1,0,149756.71,1
6,822,France,1,50,7,0.00,2,1,1,10062.80,0
7,376,Germany,0,29,4,115046.74,4,1,0,119346.88,1
8,501,France,1,44,4,142051.07,2,0,1,74940.50,0
9,684,France,1,27,2,134603.88,1,1,1,71725.73,0


In [ ]:
encod_geo= OneHotEncoder()
geo_encod= encod_geo.fit_transform(data[['Geography']]).toarray()
geo_encod

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]], shape=(10000, 3))

In [ ]:
encod_geo.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [ ]:
geo_encod_df= pd.DataFrame(
    geo_encod,
    columns= encod_geo.get_feature_names_out(
        ['Geography']
    )
)
geo_encod_df.head(10)

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
5,0.0,0.0,1.0
6,1.0,0.0,0.0
7,0.0,1.0,0.0
8,1.0,0.0,0.0
9,1.0,0.0,0.0


In [ ]:
data=pd.concat(
    [data.drop('Geography', axis=1),
    geo_encod_df
    ],axis=1

)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [ ]:
with open ('encod_gender.pkl','wb') as file:
    pickle.dump(
        encod_gender, file
    )
with open ('encod_geo.pkl','wb') as file:
    pickle.dump(
        encod_geo, file
    )
print("Success")

Success


## Step 4 - Define X and y

In [ ]:
# y = EstimatedSalary
# X = remaining useful features

x= data.drop("EstimatedSalary", axis=1)
y= data['EstimatedSalary']

print("x shape: ",x.shape)
print("y shape: ", y.shape)
print("\n input columns: ")
print (list(x.columns))


x shape:  (10000, 12)
y shape:  (10000,)

 input columns: 
['CreditScore', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'Exited', 'Geography_France', 'Geography_Germany', 'Geography_Spain']


## Step 5 - Train-test split and preprocessing

In [ ]:
# Fit encoders and scaler on training data only
xtrain, xtest, ytrain, ytest= train_test_split(
    x,y,
    test_size= 0.2,
    random_state= 42
)
print ("xtrain shape: ",xtrain.shape)
print ("xtest shape: ",xtest.shape)
print ("ytrain shape: ",ytrain.shape)
print ("ytest shape: ",ytest.shape)

xtrain shape:  (8000, 12)
xtest shape:  (2000, 12)
ytrain shape:  (8000,)
ytest shape:  (2000,)


In [ ]:
scalar= StandardScaler()
xtrain= scalar.fit_transform(xtrain)
xtest= scalar.transform(xtest)
print ("scalar xtrain shape: ", xtrain.shape)
print ("scalar xtest shape: ", xtest.shape)

scalar xtrain shape:  (8000, 12)
scalar xtest shape:  (2000, 12)


In [ ]:
with open ("scalar.pkl", "wb") as file:
    pickle.dump(scalar, file)
print ("scalar saved")

scalar saved


## Step 6 - Build a regression ANN

In [ ]:
feature_shape= (xtrain.shape[1])
print(feature_shape)

12


In [ ]:
# Example structure only
# model = keras.Sequential([
#     keras.Input(shape=(X_train.shape[1],)),
#     keras.layers.Dense(64, activation='relu'),
#     keras.layers.Dense(32, activation='relu'),
#     keras.layers.Dense(1, activation='linear')
# ])


model= keras.Sequential(
    [
        keras.Input(shape=(xtrain.shape[1],)),
        keras.layers.Dense(64, activation='relu'),
        keras.layers.Dense(32,activation= 'relu'),
        keras.layers.Dense(1,activation='linear')

    ])


In [ ]:
l1_parameters= (12*64)+64
print(l1_parameters)

832


In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

## Step 7 - Compile and train

In [ ]:
# optimizer=Adam, loss='mse' or 'mae', include a suitable regression metric
optimizer= tf.keras.optimizers.Adam (learning_rate=0.01)
loss_func= tf.keras.losses.MeanSquaredError()

In [ ]:
model.compile(
    optimizer= optimizer,
    loss= loss_func,
    metrics= ['mean_squared_error']
)

In [ ]:
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

tensorboard_callback = TensorBoard(
    log_dir=log_dir,
    histogram_freq=1
)

print("Tensorboard log directory:", log_dir)

Tensorboard log directory: logs/fit/20261101-211241


In [ ]:
early_stop= EarlyStopping(
    monitor= 'val_loss',
    patience= 20,
    restore_best_weights=True
)

In [ ]:
history= model.fit(
    xtrain, ytrain, 
    epochs= 10, 
    validation_data= (xtest, ytest),
    callbacks=[tensorboard_callback, early_stop]
)

Epoch 1/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - loss: 9633443840.0000 - mean_squared_error: 9633443840.0000 - val_loss: 3590583296.0000 - val_mean_squared_error: 3590583296.0000
Epoch 2/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 3422813184.0000 - mean_squared_error: 3422813184.0000 - val_loss: 3400330496.0000 - val_mean_squared_error: 3400330496.0000
Epoch 3/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - loss: 3351061248.0000 - mean_squared_error: 3351061248.0000 - val_loss: 3388583168.0000 - val_mean_squared_error: 3388583168.0000
Epoch 4/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - loss: 3330169088.0000 - mean_squared_error: 3330169088.0000 - val_loss: 3363600640.0000 - val_mean_squared_error: 3363600640.0000
Epoch 5/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - loss: 3319680768.0000 - mean_squared_error: 3319680768.0000 - val_loss: 3352884992.0000 - val_mean_squared_error: 3352884992.0000
Epoch 6/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step - loss: 3314265344.000

## Step 8 - Evaluate

In [ ]:
# Calculate MAE, RMSE and R2 on the untouched test set
y_pred= model.predict(xtest)
mse= mean_squared_error(ytest, y_pred)
r2= r2_score (ytest, y_pred)
print(f"Mean Squared Error: {mse:.3f}")
print(f"R Square is: {r2:.3f}")

evalu= model.evaluate (xtest, ytest)
print(evalu)


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
Mean Squared Error: 3352884657.900
R Square is: -0.016
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 3352884992.0000 - mean_squared_error: 3352884992.0000
[3352884992.0, 3352884992.0]


## Step 9 - Save and reload model

In [ ]:
# model.save('salary_model.keras')

model.save("salary_model.keras")
print("Saved model .keras file successfully.")

Saved model .keras file successfully.


In [ ]:
model=tf.keras.models.load_model ('salary_model.keras')
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,837 (34.52 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 5,892 (23.02 KB)

## Step 10 - Streamlit deployment

In [ ]:
# Build app.py, requirements.txt and deploy on Streamlit Community Cloud


## Reflection Questions
1. Why did we remove sigmoid from the output layer?
Because this model is about regression for predicting continuous value of "Estimated Salary" , not classification problem. 

2. Why is Binary Cross-Entropy not suitable here?
Because our outcome variable is not binary, it is about estimated salary prediction. 


3. What does MAE mean in salary units?
The MAE is mean absolute error which shows the average difference between predicted and actual salaries. I have calculated MSE which means average squared errors for salary prediction.

4. If RMSE is much larger than MAE, what might that suggest?
this shows the presence of outliers and model is making few large mistakes rather than small errors.

5. What are the limitations of predicting salary from this classroom dataset?
because this dataset has no missing values and outliers while in real world data, the issues of null values and outliers exist. Orignially, the data was about customer churn and our R Square is negative which is a key limitation for interperting the estimated salary. 